# LeetCode 1676 — Lowest Common Ancestor of a Binary Tree IV bằng Python

Link đề: https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree-iv/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Binary Tree

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', nodes: 'List[TreeNode]') -> 'TreeNode':
        ...
```

Mở rộng bài 236 từ **hai** nút lên **một danh sách** nút. Notebook đi từ **vét cạn kiểm tra mọi nút** → **gộp từng cặp** bằng LCA 236 → **DFS với tập mục tiêu** (dừng sớm tại mục tiêu) → **DFS đếm số mục tiêu** (bản lặp, lời giải chính) → **bảng cha + đếm lượt đi lên**.

## 1. Mục tiêu học

1. Mở rộng định nghĩa LCA cho **`k` nút**: nút thấp nhất nhận **mọi** `pᵢ` làm hậu duệ.
2. Hiểu vì sao "dừng sớm khi gặp một mục tiêu" **vẫn đúng** ở bài này (mọi mục tiêu chắc chắn tồn tại), khác với bài 1644.
3. Dùng **`set` các giá trị** để kiểm tra "nút này có phải mục tiêu không" trong `O(1)`, tránh vòng lặp `O(k)` cho mỗi nút.
4. Cài đặt DFS hậu thứ tự **lặp** (không đệ quy) đếm số mục tiêu trong cây con — an toàn với cây lệch `10⁴` nút.
5. Đối chiếu nhiều cách trên cây ngẫu nhiên với tập mục tiêu ngẫu nhiên, seed cố định.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân và một mảng các nút `nodes`, trả về **tổ tiên chung thấp nhất (LCA)** của **tất cả** các nút trong `nodes`. Mọi nút trong `nodes` đều **có trong cây**, các giá trị trong cây đều **khác nhau**.

Mở rộng định nghĩa Wikipedia: "LCA của `n` nút `p₁, p₂, …, pₙ` trong cây nhị phân `T` là nút **thấp nhất** có **mọi** `pᵢ` là hậu duệ (cho phép một nút là hậu duệ của chính nó)."

Cây dùng cho cả 3 ví dụ: `root = [3,5,1,6,2,0,8,null,null,7,4]`

```
            3
          /   \
         5     1
        / \   / \
       6   2 0   8
          / \
         7   4
```

**Ví dụ 1:** `nodes = [4,7]` → `2`.

**Ví dụ 2:** `nodes = [1]` → `1` (LCA của một nút là chính nó).

**Ví dụ 3:** `nodes = [7,6,2,4]` → `5`.

**Ràng buộc:**
- Số nút trong `[1, 10⁴]`; `-10⁹ ≤ Node.val ≤ 10⁹`; mọi giá trị khác nhau.
- Mọi `nodes[i]` có trong cây và đôi một khác nhau.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values):
    """Dựng cây từ level-order kiểu LeetCode; trả về (root, dict val -> node)."""
    if not values or values[0] is None:
        return None, {}
    root = TreeNode(values[0]); index = {root.val: root}
    q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i]); setattr(node, side, child)
                index[child.val] = child; q.append(child)
            i += 1
    return root, index

def to_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

TREE = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
examples = [(TREE, [4, 7], 2), (TREE, [1], 1), (TREE, [7, 6, 2, 4], 5)]
root, idx = build_tree(TREE)
print("khứ hồi:", to_list(root))
for vals, targets, exp in examples:
    print(f"nodes={targets} -> kỳ vọng {exp}")

khứ hồi: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
nodes=[4, 7] -> kỳ vọng 2
nodes=[1] -> kỳ vọng 1
nodes=[7, 6, 2, 4] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Hậu duệ và cây con

`y` là hậu duệ của `x` ⇔ `y` nằm trong **cây con gốc `x`**. "LCA của tập `S`" = nút **sâu nhất** mà cây con của nó chứa toàn bộ `S`.

In [2]:
def subtree_vals(node):
    out, stack = [], [node]
    while stack:
        u = stack.pop()
        out.append(u.val)
        if u.right: stack.append(u.right)
        if u.left: stack.append(u.left)
    return out

for v in [5, 2, 1]:
    print(f"cây con của {v}: {sorted(subtree_vals(idx[v]))}")

cây con của 5: [2, 4, 5, 6, 7]
cây con của 2: [2, 4, 7]
cây con của 1: [0, 1, 8]


### 3.2 `set` cho tập mục tiêu

Chuyển `nodes` thành `{node.val for node in nodes}` để kiểm `u.val in targets` trong `O(1)`. So sánh theo `val` an toàn vì giá trị trong cây là duy nhất.

In [3]:
nodes = [idx[7], idx[6], idx[2], idx[4]]
targets = {u.val for u in nodes}
print(targets, 7 in targets, 5 in targets)

{2, 4, 6, 7} True False


### 3.3 LCA của hai nút (bài 236) — viên gạch cơ bản

Khi **chắc chắn** `p`, `q` có trong cây: nếu nút hiện tại là `p` hoặc `q` thì trả về nó; nếu cả hai cây con trả về khác `None` thì nút hiện tại là LCA.

In [4]:
def lca2(node, p, q):
    if node is None or node.val == p.val or node.val == q.val:
        return node
    l, r = lca2(node.left, p, q), lca2(node.right, p, q)
    if l and r:
        return node
    return l or r

print("LCA(7, 4) =", lca2(root, idx[7], idx[4]).val, "| LCA(6, 8) =", lca2(root, idx[6], idx[8]).val)

LCA(7, 4) = 2 | LCA(6, 8) = 3


### 3.4 Hậu thứ tự lặp với stack `(node, done)`

Đẩy `(node, False)`; khi lấy ra lần đầu, đẩy lại `(node, True)` rồi hai con. Khi lấy ra với `done = True`, hai con đã xử lý xong ⇒ tổng hợp kết quả. Không đệ quy ⇒ không lo giới hạn `~1000` tầng.

In [5]:
order = []
stack = [(root, False)]
while stack:
    node, done = stack.pop()
    if node is None:
        continue
    if done:
        order.append(node.val)
    else:
        stack.append((node, True)); stack.append((node.right, False)); stack.append((node.left, False))
print("hậu thứ tự (lặp):", order)

hậu thứ tự (lặp): [6, 7, 4, 2, 5, 0, 8, 1, 3]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3 với `S = {7, 6, 2, 4}`. Viết đường đi từ gốc:
- `7`: `3 → 5 → 2 → 7`
- `6`: `3 → 5 → 6`
- `2`: `3 → 5 → 2`
- `4`: `3 → 5 → 2 → 4`

Tiền tố chung của **mọi** đường: `3 → 5` ⇒ LCA = `5`.

Một cách nhìn khác: đếm số mục tiêu trong cây con của mỗi nút. Nút sâu nhất có số đếm bằng `|S|` là LCA.

| nút | mục tiêu trong cây con | đếm |
|---|---|---|
| 7, 4 | chính nó | 1 |
| 2 | 2, 7, 4 | 3 |
| 6 | 6 | 1 |
| **5** | 6, 2, 7, 4 | **4 = \|S\|** |
| 3 | 6, 2, 7, 4 | 4 |

In [6]:
def path_to(root, val):
    parent = {root: None}; stack = [root]; target = None
    while stack:
        u = stack.pop()
        if u.val == val:
            target = u; break
        for c in (u.left, u.right):
            if c: parent[c] = u; stack.append(c)
    path = []
    while target:
        path.append(target.val); target = parent[target]
    return path[::-1]

S = [7, 6, 2, 4]
paths = [path_to(root, v) for v in S]
for v, p in zip(S, paths):
    print(f"đường tới {v}: {p}")
common = []
for col in zip(*paths):
    if len(set(col)) != 1: break
    common.append(col[0])
print("tiền tố chung:", common, "-> LCA =", common[-1])

đường tới 7: [3, 5, 2, 7]
đường tới 6: [3, 5, 6]
đường tới 2: [3, 5, 2]
đường tới 4: [3, 5, 2, 4]
tiền tố chung: [3, 5] -> LCA = 5


## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi nút

**Ý tưởng:** với mỗi nút `x`, liệt kê cây con của nó và kiểm tra có chứa mọi mục tiêu không; lấy nút **sâu nhất** thỏa mãn.

In [7]:
def lca_brute(root, nodes):
    targets = {u.val for u in nodes}
    best, best_d = None, -1
    stack = [(root, 0)]
    while stack:
        x, d = stack.pop()
        if d > best_d and targets <= set(subtree_vals(x)):
            best, best_d = x, d
        for c in (x.left, x.right):
            if c: stack.append((c, d + 1))
    return best

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    print(f"nodes={tv} -> {lca_brute(r, [ix[v] for v in tv]).val} (kỳ vọng {exp})")

nodes=[4, 7] -> 2 (kỳ vọng 2)
nodes=[1] -> 1 (kỳ vọng 1)
nodes=[7, 6, 2, 4] -> 5 (kỳ vọng 5)


**Độ phức tạp:** mỗi nút liệt kê cây con của nó ⇒ `Σ |cây con| = Σ (độ sâu + 1)` = `O(n·h)`, xấu nhất `O(n²)`. Bộ nhớ `O(n)`.

**Nút thắt:** cây con bị quét lại cho mọi tổ tiên. Ta muốn mỗi nút được thăm **một lần** và "báo cáo" lên cha.

## 6. Ý tưởng thuật toán / bước đột phá

**Cách nghĩ 1 — LCA có tính kết hợp:** `LCA(p₁, …, pₖ) = LCA(LCA(p₁, …, pₖ₋₁), pₖ)`. Gộp lần lượt từng nút ⇒ `k − 1` lần gọi LCA hai nút, mỗi lần `O(n)` ⇒ `O(n·k)`.

**Cách nghĩ 2 — một lần DFS, tổng quát bài 236:** `dfs(x)` trả về
- `x` nếu `x` là mục tiêu (dừng sớm — **hợp lệ** vì mọi mục tiêu đều có mặt: nếu có mục tiêu khác dưới `x`, LCA vẫn là `x` hoặc tổ tiên của `x`… chi tiết ở mục 10),
- nút gặp được nếu chỉ một cây con trả về khác `None`,
- `x` nếu **cả hai** cây con trả về khác `None` (các mục tiêu nằm ở hai phía).

**Cách nghĩ 3 — đếm:** `cnt(x)` = số mục tiêu trong cây con của `x` = `cnt(left) + cnt(right) + [x ∈ S]`. Theo hậu thứ tự, nút **đầu tiên** đạt `cnt = |S|` là LCA.

**Quy tắc chọn:** nếu cần chắc chắn và không đệ quy ⇒ đếm hậu thứ tự lặp (cách 4). Nếu chỉ cần code ngắn ⇒ DFS tập mục tiêu (cách 3).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Gộp từng cặp bằng LCA hai nút

In [8]:
def lca_fold(root, nodes):
    cur = nodes[0]
    for u in nodes[1:]:
        cur = lca2(root, cur, u)
    return cur

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    print(f"nodes={tv} -> {lca_fold(r, [ix[v] for v in tv]).val} (kỳ vọng {exp})")

nodes=[4, 7] -> 2 (kỳ vọng 2)
nodes=[1] -> 1 (kỳ vọng 1)
nodes=[7, 6, 2, 4] -> 5 (kỳ vọng 5)


**Độ phức tạp:** `O(n · k)` thời gian, `O(h)` stack. Với `k` lớn (tới `10⁴`) có thể tới `10⁸` — chậm.

### 7.2 Cách 3 — DFS với tập mục tiêu (tổng quát bài 236)

In [9]:
def lca_set_dfs(root, nodes):
    targets = {u.val for u in nodes}
    def dfs(x):
        if x is None or x.val in targets:
            return x
        l, r = dfs(x.left), dfs(x.right)
        if l and r:
            return x
        return l or r
    return dfs(root)

class SolutionRecursive:
    def lowestCommonAncestor(self, root: 'TreeNode', nodes: 'List[TreeNode]') -> 'TreeNode':
        targets = {u.val for u in nodes}
        def dfs(x):
            if x is None or x.val in targets:
                return x
            l, r = dfs(x.left), dfs(x.right)
            return x if l and r else (l or r)
        return dfs(root)

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    ns = [ix[v] for v in tv]
    print(f"nodes={tv} -> {lca_set_dfs(r, ns).val}, {SolutionRecursive().lowestCommonAncestor(r, ns).val} (kỳ vọng {exp})")

nodes=[4, 7] -> 2, 2 (kỳ vọng 2)
nodes=[1] -> 1, 1 (kỳ vọng 1)
nodes=[7, 6, 2, 4] -> 5, 5 (kỳ vọng 5)


**Độ phức tạp:** `O(n + k)` thời gian, `O(h + k)` bộ nhớ. Đệ quy ⇒ cây lệch `10⁴` nút có thể `RecursionError` trong Python.

### 7.3 Cách 4 — Đếm số mục tiêu, hậu thứ tự lặp (lời giải chính)

In [10]:
def lca_count_iter(root, nodes):
    targets = {u.val for u in nodes}
    k = len(targets)
    cnt = {None: 0}
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if not done:
            stack.append((x, True)); stack.append((x.right, False)); stack.append((x.left, False))
        else:
            c = cnt[x.left] + cnt[x.right] + (x.val in targets)
            if c == k:
                return x                 # nút đầu tiên (sâu nhất) chứa đủ mục tiêu
            cnt[x] = c
    return None

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    print(f"nodes={tv} -> {lca_count_iter(r, [ix[v] for v in tv]).val} (kỳ vọng {exp})")

nodes=[4, 7] -> 2 (kỳ vọng 2)
nodes=[1] -> 1 (kỳ vọng 1)
nodes=[7, 6, 2, 4] -> 5 (kỳ vọng 5)


**Độ phức tạp:** `O(n + k)` thời gian, `O(n)` bộ nhớ (stack + `cnt`). Không đệ quy. Có thể dừng sớm ngay khi gặp nút đạt đủ `k`.

### 7.4 Cách 5 — Bảng cha + đếm lượt đi lên

1. BFS dựng `parent` và `depth`.
2. Từ mỗi mục tiêu đi lên tới gốc, tăng `visits[x]` cho mọi tổ tiên `x` (gồm chính nó).
3. LCA = nút **sâu nhất** có `visits[x] == k`.

Đơn giản nhưng `O(k · h)` — xấu nhất `O(n²)` khi cây lệch và `k` lớn. Có thể tối ưu: dừng đi lên khi gặp nút đã được thăm bởi mục tiêu khác? (Không đúng cho việc đếm — xem bài tập.)

In [11]:
def lca_parent_count(root, nodes):
    parent, depth = {root: None}, {root: 0}
    q = deque([root])
    while q:
        u = q.popleft()
        for c in (u.left, u.right):
            if c:
                parent[c] = u; depth[c] = depth[u] + 1; q.append(c)
    by_val = {u.val: u for u in parent}
    visits = {}
    for t in {u.val for u in nodes}:
        x = by_val[t]
        while x is not None:
            visits[x] = visits.get(x, 0) + 1
            x = parent[x]
    k = len({u.val for u in nodes})
    return max((x for x, v in visits.items() if v == k), key=lambda x: depth[x])

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    print(f"nodes={tv} -> {lca_parent_count(r, [ix[v] for v in tv]).val} (kỳ vọng {exp})")

nodes=[4, 7] -> 2 (kỳ vọng 2)
nodes=[1] -> 1 (kỳ vọng 1)
nodes=[7, 6, 2, 4] -> 5 (kỳ vọng 5)


**Độ phức tạp:** `O(n + k·h)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 4 (đếm lặp) cho ví dụ 3, `S = {7, 6, 2, 4}`, `k = 4`. Thứ tự hậu thứ tự: `6, 7, 4, 2, 5, …`

| nút | cnt trái | cnt phải | là mục tiêu? | cnt | == 4? |
|---|---|---|---|---|---|
| 6 | 0 | 0 | ✔ | 1 | |
| 7 | 0 | 0 | ✔ | 1 | |
| 4 | 0 | 0 | ✔ | 1 | |
| 2 | 1 | 1 | ✔ | 3 | |
| 5 | 1 | 3 | | 4 | **✔ trả về 5** |

In [12]:
def trace_count(values, target_vals):
    r, ix = build_tree(values)
    targets = set(target_vals); k = len(targets)
    cnt = {None: 0}
    stack = [(r, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if not done:
            stack.append((x, True)); stack.append((x.right, False)); stack.append((x.left, False))
        else:
            me = int(x.val in targets)
            c = cnt[x.left] + cnt[x.right] + me
            print(f"  nút {x.val}: trái={cnt[x.left]}, phải={cnt[x.right]}, mục tiêu={me}, cnt={c}{'  == k -> LCA' if c == k else ''}")
            if c == k:
                return x
            cnt[x] = c

print("S = {7, 6, 2, 4}:"); trace_count(TREE, [7, 6, 2, 4])
print("S = {0, 4}:"); trace_count(TREE, [0, 4])

S = {7, 6, 2, 4}:
  nút 6: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 7: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 4: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 2: trái=1, phải=1, mục tiêu=1, cnt=3
  nút 5: trái=1, phải=3, mục tiêu=0, cnt=4  == k -> LCA
S = {0, 4}:
  nút 6: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 7: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 4: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 2: trái=0, phải=1, mục tiêu=0, cnt=1
  nút 5: trái=0, phải=1, mục tiêu=0, cnt=1
  nút 0: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 8: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 1: trái=1, phải=0, mục tiêu=0, cnt=1
  nút 3: trái=1, phải=1, mục tiêu=0, cnt=2  == k -> LCA


TreeNode(3)

Trace cách 3 (DFS tập mục tiêu) cho ví dụ 1, in giá trị trả về của mỗi lời gọi:

In [13]:
def trace_set_dfs(values, target_vals):
    r, ix = build_tree(values)
    targets = set(target_vals)
    def dfs(x, d):
        if x is None:
            return None
        if x.val in targets:
            print("  " * d + f"dfs({x.val}) là mục tiêu -> trả {x.val} (không xuống tiếp)")
            return x
        l, rr = dfs(x.left, d + 1), dfs(x.right, d + 1)
        res = x if (l and rr) else (l or rr)
        print("  " * d + f"dfs({x.val}): trái={l.val if l else None}, phải={rr.val if rr else None} -> trả {res.val if res else None}")
        return res
    return dfs(r, 0)

print("LCA =", trace_set_dfs(TREE, [4, 7]).val)

    dfs(6): trái=None, phải=None -> trả None
      dfs(7) là mục tiêu -> trả 7 (không xuống tiếp)
      dfs(4) là mục tiêu -> trả 4 (không xuống tiếp)
    dfs(2): trái=7, phải=4 -> trả 2
  dfs(5): trái=None, phải=2 -> trả 2
    dfs(0): trái=None, phải=None -> trả None
    dfs(8): trái=None, phải=None -> trả None
  dfs(1): trái=None, phải=None -> trả None
dfs(3): trái=2, phải=None -> trả 2
LCA = 2


## 9. Code nộp LeetCode

Lời giải chính: hậu thứ tự lặp đếm số mục tiêu (một lượt, không đệ quy, an toàn với cây lệch).

In [14]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', nodes: 'List[TreeNode]') -> 'TreeNode':
        targets = {u.val for u in nodes}
        k = len(targets)
        cnt = {None: 0}                       # cnt[x] = số mục tiêu trong cây con gốc x
        stack = [(root, False)]
        while stack:
            x, done = stack.pop()
            if x is None:
                continue
            if not done:
                stack.append((x, True))
                stack.append((x.right, False))
                stack.append((x.left, False))
            else:
                c = cnt[x.left] + cnt[x.right] + (x.val in targets)
                if c == k:
                    return x
                cnt[x] = c
        return None

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    assert Solution().lowestCommonAncestor(r, [ix[v] for v in tv]).val == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `S` là tập mục tiêu, `k = |S|`, `cnt(x) = |S ∩ cây_con(x)|`.

**Cách 4 (đếm).** `cnt` được tính đúng theo hậu thứ tự (hai con trước). Tập `A = {x : cnt(x) = k}` là tập các **tổ tiên chung** của mọi phần tử `S`; nó là một **đường** từ gốc xuống (cha của phần tử thuộc `A` cũng thuộc `A`), và LCA là phần tử **sâu nhất** của `A`. Trong hậu thứ tự, mọi nút của cây con `x` được xử lý trước `x`, nên phần tử sâu nhất của `A` được gặp **đầu tiên** ⇒ trả về đúng.

**Cách 3 (dừng sớm).** Gọi `L` là LCA. Bổ đề: `dfs(x)` trả về `None` nếu cây con `x` không chứa mục tiêu; trả về `L` nếu cây con `x` chứa `L`… Cụ thể:
- Nếu `L ∈ S` (một mục tiêu là tổ tiên của tất cả): mọi mục tiêu khác nằm trong cây con `L`. DFS tới `L` thì trả `L` ngay (không cần xuống), các nhánh không chứa `L` không chứa mục tiêu nào ⇒ trả `None`; tổ tiên của `L` chỉ nhận được `L` từ đúng một phía ⇒ truyền `L` lên gốc.
- Nếu `L ∉ S`: hai cây con của `L` đều chứa mục tiêu (nếu không thì con chứa hết mục tiêu sẽ là tổ tiên chung thấp hơn). Mỗi cây con trả về khác `None` ⇒ `dfs(L) = L`; tổ tiên của `L` chỉ nhận `L` từ một phía ⇒ truyền lên.
Điều này **cần** mọi mục tiêu tồn tại — đúng theo đề (khác bài 1644).

**Cách 2 (gộp cặp).** LCA có tính kết hợp: tổ tiên chung của `{p₁..pₖ}` = tổ tiên chung của `{LCA(p₁..pₖ₋₁), pₖ}`, vì một nút là tổ tiên của `p₁..pₖ₋₁` ⇔ là tổ tiên của `LCA(p₁..pₖ₋₁)`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `k = len(nodes)`, `h` là chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n·h)` (xấu nhất `O(n²)`) | `O(n)` |
| 2. Gộp cặp | `O(n·k)` | `O(h)` |
| 3. DFS tập mục tiêu | `O(n + k)` | `O(h + k)` (đệ quy) |
| 4. Đếm hậu thứ tự lặp | `O(n + k)` | `O(n)` |
| 5. Bảng cha + đếm | `O(n + k·h)` | `O(n)` |

In [15]:
def random_tree(n, rnd):
    vals = rnd.sample(range(-10**9, 10**9), n)
    nodes = [TreeNode(vals[0])]
    slots = [(nodes[0], 'left'), (nodes[0], 'right')]
    for v in vals[1:]:
        j = rnd.randrange(len(slots)); par, side = slots[j]
        slots[j] = slots[-1]; slots.pop()
        ch = TreeNode(v); setattr(par, side, ch); nodes.append(ch)
        slots += [(ch, 'left'), (ch, 'right')]
    return nodes[0], nodes

def skewed(n):
    root = TreeNode(0); cur = root; nodes = [root]
    for v in range(1, n):
        cur.right = TreeNode(v); cur = cur.right; nodes.append(cur)
    return root, nodes

rnd = random.Random(1676)
R, nodes_all = random_tree(10**4, rnd)
T = rnd.sample(nodes_all, 300)
for name, fn in [("fold", lca_fold), ("set_dfs", lca_set_dfs), ("count_iter", lca_count_iter), ("parent", lca_parent_count), ("Solution", Solution().lowestCommonAncestor)]:
    t0 = time.perf_counter(); a = fn(R, T); dt = time.perf_counter() - t0
    print(f"cây ngẫu nhiên, k=300  {name:<10}: LCA={a.val:>11}  {dt * 1000:7.1f} ms")
S_root, S_nodes = skewed(10**4)
T2 = S_nodes[5000:5010]
for name, fn in [("count_iter", lca_count_iter), ("Solution", Solution().lowestCommonAncestor)]:
    t0 = time.perf_counter(); a = fn(S_root, T2); dt = time.perf_counter() - t0
    print(f"cây lệch 10^4          {name:<10}: LCA={a.val:>11}  {dt * 1000:7.1f} ms")

cây ngẫu nhiên, k=300  fold      : LCA= -940817413      2.1 ms
cây ngẫu nhiên, k=300  set_dfs   : LCA= -940817413      1.5 ms
cây ngẫu nhiên, k=300  count_iter: LCA= -940817413      4.9 ms
cây ngẫu nhiên, k=300  parent    : LCA= -940817413      5.9 ms
cây ngẫu nhiên, k=300  Solution  : LCA= -940817413      4.6 ms
cây lệch 10^4          count_iter: LCA=       5000      3.2 ms
cây lệch 10^4          Solution  : LCA=       5000      2.9 ms


## 12. Bộ test

In [16]:
ALL = {
    "brute": lca_brute,
    "fold": lca_fold,
    "set_dfs": lca_set_dfs,
    "count_iter": lca_count_iter,
    "parent": lca_parent_count,
    "SolRec": lambda r, ns: SolutionRecursive().lowestCommonAncestor(r, ns),
    "Solution": lambda r, ns: Solution().lowestCommonAncestor(r, ns),
}

def check(r, ns, exp=None):
    res = {name: fn(r, ns) for name, fn in ALL.items()}
    first = res["brute"]
    assert all(v is first for v in res.values()), {k: v.val for k, v in res.items()}
    if exp is not None:
        assert first.val == exp
    return first.val

for vals, tv, exp in examples:
    r, ix = build_tree(vals)
    check(r, [ix[v] for v in tv], exp)
    check(r, [ix[v] for v in reversed(tv)], exp)          # thứ tự trong nodes không quan trọng
print("Ví dụ đề: OK")

r, ix = build_tree(TREE)
check(r, [ix[3]], 3)                                       # chỉ gốc
check(r, [ix[v] for v in [6, 7, 4, 0, 8]], 3)              # trải hai nhánh
check(r, [ix[v] for v in ix], 3)                           # tất cả các nút
check(r, [ix[2], ix[4]], 2)                                # một mục tiêu là tổ tiên của mục tiêu khác
check(r, [ix[0], ix[8], ix[1]], 1)
r1, ix1 = build_tree([1])
check(r1, [ix1[1]], 1)                                     # cây một nút
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
rnd = random.Random(2014)
for _ in range(2500):
    n = rnd.randint(1, 18)
    r, nodes_all = random_tree(n, rnd)
    k = rnd.randint(1, n)
    ns = rnd.sample(nodes_all, k)
    check(r, ns)
print("2500 cây ngẫu nhiên nhỏ, tập mục tiêu ngẫu nhiên: OK")

S_root, S_nodes = skewed(10**4)
assert Solution().lowestCommonAncestor(S_root, S_nodes[100:200]).val == 100
assert lca_count_iter(S_root, [S_nodes[-1]]).val == 10**4 - 1
print("Cây lệch 10^4 nút (bản lặp): OK")

2500 cây ngẫu nhiên nhỏ, tập mục tiêu ngẫu nhiên: OK


Cây lệch 10^4 nút (bản lặp): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lấy LCA của hai nút đầu

In [18]:
def lca_bug_first_two(root, nodes):
    if len(nodes) == 1:
        return nodes[0]
    return lca2(root, nodes[0], nodes[1])                 # SAI: bỏ qua các nút còn lại

r, ix = build_tree(TREE)
ns = [ix[v] for v in [7, 4, 6]]
print("hai nút đầu:", lca_bug_first_two(r, ns), "| đúng:", Solution().lowestCommonAncestor(r, ns))

hai nút đầu: TreeNode(2) | đúng: TreeNode(5)


**Vì sao sai:** LCA của tập phải là tổ tiên của **mọi** phần tử. `LCA(7, 4) = 2` không phải tổ tiên của `6`. **Sửa:** gộp mọi nút (cách 2) hoặc dùng tập mục tiêu (cách 3/4).

### Lỗi 2 — Đếm số mục tiêu nhưng dừng sớm khi gặp mục tiêu

In [19]:
def lca_bug_count_early(root, nodes):
    targets = {u.val for u in nodes}; k = len(targets)
    ans = None
    def dfs(x):
        nonlocal ans
        if x is None:
            return 0
        if x.val in targets:
            return 1                                    # SAI: không đếm các mục tiêu bên dưới x
        c = dfs(x.left) + dfs(x.right)
        if c == k and ans is None:
            ans = x
        return c
    dfs(root)
    return ans

r, ix = build_tree(TREE)
ns = [ix[v] for v in [7, 6, 2, 4]]
print("đếm + dừng sớm:", lca_bug_count_early(r, ns), "| đúng:", Solution().lowestCommonAncestor(r, ns))

đếm + dừng sớm: None | đúng: TreeNode(5)


**Vì sao sai:** kiểu "dừng sớm" chỉ hợp với cách 3 (trả về **nút**), không hợp với cách **đếm**: `2` là mục tiêu và còn `7`, `4` bên dưới — dừng ở `2` chỉ đếm được 1 thay vì 3, nên không nút nào đạt `k = 4`. **Sửa:** khi đếm, luôn xuống cả hai con rồi cộng `[x là mục tiêu]`.

### Lỗi 3 — Ghi đè đáp án ở mọi nút có `cnt == k`

In [20]:
def lca_bug_overwrite(root, nodes):
    targets = {u.val for u in nodes}; k = len(targets)
    ans = None
    def dfs(x):
        nonlocal ans
        if x is None:
            return 0
        c = dfs(x.left) + dfs(x.right) + (x.val in targets)
        if c == k:
            ans = x                                     # SAI: tổ tiên ghi đè
        return c
    dfs(root)
    return ans

r, ix = build_tree(TREE)
ns = [ix[v] for v in [4, 7]]
print("ghi đè:", lca_bug_overwrite(r, ns), "| đúng:", Solution().lowestCommonAncestor(r, ns))

ghi đè: TreeNode(3) | đúng: TreeNode(2)


**Sửa:** chỉ ghi lần đầu (`ans is None`) hoặc trả về ngay như bản lặp.

### Lỗi 4 — Kiểm tra mục tiêu bằng list (`x in nodes`) — chậm `O(k)` mỗi nút

In [21]:
def lca_slow_list(root, nodes):
    target_list = [u.val for u in nodes]                 # list: x.val in list tốn O(k)
    def dfs(x):
        if x is None or x.val in target_list:
            return x
        l, r = dfs(x.left), dfs(x.right)
        return x if l and r else (l or r)
    return dfs(root)

rnd = random.Random(9)
R, nodes_all = random_tree(6000, rnd)
ns = [u for u in nodes_all if u.left is None and u.right is None]     # mọi lá: DFS phải đi hết cây
t0 = time.perf_counter(); a = lca_slow_list(R, ns); t_list = time.perf_counter() - t0
t0 = time.perf_counter(); b = lca_set_dfs(R, ns); t_set = time.perf_counter() - t0
print(f"k = {len(ns)} mục tiêu | list: {t_list * 1000:.1f} ms | set: {t_set * 1000:.1f} ms | cùng kết quả: {a is b}")

k = 2007 mục tiêu | list: 43.4 ms | set: 0.6 ms | cùng kết quả: True


**Vì sao "sai":** đúng kết quả nhưng `O(n · k)` — với `n = k = 10⁴` là `10⁸` phép so sánh. **Sửa:** chuyển sang `set`.

### Lỗi 5 — Đệ quy trên cây lệch

In [22]:
S_root, S_nodes = skewed(10**4)
try:
    lca_set_dfs(S_root, S_nodes[-3:])
    print("Đệ quy chạy được (giới hạn đệ quy đã được nâng)")
except RecursionError:
    print("RecursionError với cây lệch 10^4 nút (giới hạn =", sys.getrecursionlimit(), ")")
print("Bản lặp:", Solution().lowestCommonAncestor(S_root, S_nodes[-3:]))

RecursionError với cây lệch 10^4 nút (giới hạn = 1000 )
Bản lặp: TreeNode(9997)


**Sửa:** dùng hậu thứ tự lặp (cách 4) hoặc BFS + bảng cha.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "LCA của **nhiều** nút" ⇒ hoặc **gộp cặp** (tính kết hợp), hoặc **một DFS với tập mục tiêu**.
- Đề bảo đảm "mọi nút đều tồn tại" ⇒ được phép **dừng sớm** khi gặp mục tiêu (kiểu bài 236). Không bảo đảm ⇒ phải **đếm** (kiểu bài 1644).
- Kiểm tra thành viên ⇒ luôn chuyển sang `set`.
- Cây `10⁴` nút trong Python ⇒ chuẩn bị sẵn khuôn hậu thứ tự lặp `(node, done)`.
- LCA của một nút là chính nó — kiểm tra trường hợp `k = 1`.

## 15. Biến thể của bài

1. **Một số nút có thể không tồn tại** (kết hợp với 1644): đếm số mục tiêu thấy được; nếu nhỏ hơn `k` thì `None`.
2. **Cây có `parent` và không có gốc** (kết hợp với 1650): gộp cặp bằng hai con trỏ đổi đường.
3. **Nhiều truy vấn LCA tập** trên cùng cây: tiền xử lý Euler tour; LCA của tập = LCA của nút có thời điểm vào **nhỏ nhất** và **lớn nhất**.
4. **Cây BST:** LCA của tập = LCA của giá trị **min** và **max** trong tập.
5. **Đường đi nhỏ nhất (cây Steiner) nối mọi nút trong tập:** tổng khoảng cách theo thứ tự Euler / 2.

In [23]:
def euler_lca_of_set(root, nodes):
    """Biến thể 3: LCA của tập = LCA(nút có tin nhỏ nhất, nút có tin lớn nhất) theo thứ tự tiền thứ tự."""
    tin = {}
    stack, t = [root], 0
    while stack:
        u = stack.pop()
        tin[u] = t; t += 1
        if u.right: stack.append(u.right)
        if u.left: stack.append(u.left)
    lo = min(nodes, key=lambda u: tin[u]); hi = max(nodes, key=lambda u: tin[u])
    return Solution().lowestCommonAncestor(root, [lo, hi])

def lca_maybe_missing(root, values):
    """Biến thể 1: giá trị có thể không có trong cây -> None nếu thiếu."""
    targets = set(values); k = len(targets)
    cnt = {None: 0}; stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None: continue
        if not done:
            stack += [(x, True), (x.right, False), (x.left, False)]
        else:
            c = cnt[x.left] + cnt[x.right] + (x.val in targets)
            if c == k: return x
            cnt[x] = c
    return None

rnd = random.Random(15)
for _ in range(800):
    r, nodes_all = random_tree(rnd.randint(1, 20), rnd)
    ns = rnd.sample(nodes_all, rnd.randint(1, len(nodes_all)))
    assert euler_lca_of_set(r, ns) is Solution().lowestCommonAncestor(r, ns)
r, ix = build_tree(TREE)
print("Euler: OK | LCA tập có giá trị vắng mặt [7, 99]:", lca_maybe_missing(r, [7, 99]), "| [7, 8]:", lca_maybe_missing(r, [7, 8]))

Euler: OK | LCA tập có giá trị vắng mặt [7, 99]: None | [7, 8]: TreeNode(3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | nút sâu nhất có cây con ⊇ S | `O(n²)` | `O(n)` | đối chiếu |
| 2. Gộp cặp | LCA kết hợp | `O(n·k)` | `O(h)` | `k` nhỏ, đã có hàm LCA 2 nút |
| 3. DFS tập mục tiêu | 236 tổng quát, dừng sớm | `O(n + k)` | `O(h + k)` | code ngắn, cây nông |
| 4. Đếm lặp | nút đầu tiên `cnt = k` | `O(n + k)` | `O(n)` | **nộp bài**, an toàn |
| 5. Bảng cha + đếm | đếm lượt đi lên | `O(n + k·h)` | `O(n)` | khi đã có `parent` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Trên cây ví dụ, LCA của `[0, 8]`, `[6, 4]`, `[7, 1]` là gì?

**Bài 2 (tính tay):** Ghi giá trị `cnt` theo hậu thứ tự cho `S = {6, 0}` và cho biết nút nào được trả về.

**Bài 3 (điền TODO):** Hoàn thiện `lca_values(root, values)` nhận **danh sách giá trị** (không phải nút) và trả về **giá trị** LCA, dùng `Solution`.

**Bài 4 (tự viết lại):** Viết lại cách 3 (DFS tập mục tiêu) dưới dạng **lặp** (hậu thứ tự với stack), trả về nút giống hệt bản đệ quy.

**Bài 5 (mini challenge):** Viết `steiner_size(root, nodes)` trả về **số cạnh** của cây con nhỏ nhất nối mọi nút trong `nodes` (gồm LCA). Gợi ý: một cạnh `(cha, con)` thuộc cây Steiner ⇔ cây con của `con` chứa ít nhất một mục tiêu **và** không chứa tất cả mục tiêu… hoặc con nằm dưới LCA và có mục tiêu.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def lca_values(root, values):
    # TODO: tìm các nút có giá trị trong values (BFS), gọi Solution, trả .val
    pass

# Bài 4 — TODO
def lca_set_iter(root, nodes):
    # TODO: ret[x] = kết quả dfs(x); hậu thứ tự lặp; nút mục tiêu: ret = x (không cần xử lý con)
    pass

# Bài 5 — TODO
def steiner_size(root, nodes):
    # TODO: tính cnt bằng hậu thứ tự; đếm các nút x != LCA trong cây con của LCA có cnt[x] > 0
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `0`, `8` là hai con của `1`.
- Mức 2: `6`, `4` đều dưới `5`.
- Mức 3: `7` ở nhánh trái của gốc, `1` ở nhánh phải.

**Bài 2**
- Mức 1: `6` dưới `5`, `0` dưới `1`.
- Mức 2: `cnt(5) = 1`, `cnt(1) = 1`.
- Mức 3: Gốc `3` là nút đầu tiên đạt `cnt = 2`.

**Bài 3**
- Mức 1: Dựng `dict` giá trị → nút bằng BFS.
- Mức 2: Tạo list nút từ `values`.
- Mức 3: Trả `.val` của kết quả.

**Bài 4**
- Mức 1: Dùng `ret = {None: None}`.
- Mức 2: Khi lấy ra lần đầu một nút mục tiêu, có thể gán `ret[x] = x` ngay mà không đẩy con.
- Mức 3: Nút thường: `ret[x] = x if ret[l] and ret[r] else (ret[l] or ret[r])`.

**Bài 5**
- Mức 1: Cây Steiner gồm LCA và mọi nút nằm trên đường từ LCA tới một mục tiêu.
- Mức 2: Nút `x` (khác LCA, trong cây con của LCA) thuộc cây Steiner ⇔ `cnt(x) > 0`.
- Mức 3: Số cạnh = số nút của cây Steiner − 1.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r, ix = build_tree(TREE)
res1 = [Solution().lowestCommonAncestor(r, [ix[a], ix[b]]).val for a, b in [(0, 8), (6, 4), (7, 1)]]
print("Bài 1:", res1)
assert res1 == [1, 5, 3]

# Bài 2
ans2 = trace_count(TREE, [6, 0])
assert ans2.val == 3

# Bài 3
def lca_values_sol(root, values):
    by_val = {}
    q = deque([root])
    while q:
        u = q.popleft(); by_val[u.val] = u
        for c in (u.left, u.right):
            if c: q.append(c)
    return Solution().lowestCommonAncestor(root, [by_val[v] for v in values]).val

print("Bài 3:", lca_values_sol(r, [7, 6, 2, 4]), lca_values_sol(r, [1]))
assert lca_values_sol(r, [7, 6, 2, 4]) == 5 and lca_values_sol(r, [1]) == 1

Bài 1: [1, 5, 3]
  nút 6: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 7: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 4: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 2: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 5: trái=1, phải=0, mục tiêu=0, cnt=1
  nút 0: trái=0, phải=0, mục tiêu=1, cnt=1
  nút 8: trái=0, phải=0, mục tiêu=0, cnt=0
  nút 1: trái=1, phải=0, mục tiêu=0, cnt=1
  nút 3: trái=1, phải=1, mục tiêu=0, cnt=2  == k -> LCA
Bài 3: 5 1


In [26]:
# Bài 4
def lca_set_iter_sol(root, nodes):
    targets = {u.val for u in nodes}
    ret = {None: None}
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if x.val in targets:
            ret[x] = x                       # mục tiêu: trả chính nó, không xuống tiếp
            continue
        if not done:
            stack += [(x, True), (x.right, False), (x.left, False)]
        else:
            l, rr = ret.get(x.left), ret.get(x.right)
            ret[x] = x if (l and rr) else (l or rr)
    return ret[root]

# Bài 5
def steiner_size_sol(root, nodes):
    targets = {u.val for u in nodes}
    L = Solution().lowestCommonAncestor(root, nodes)
    cnt = {None: 0}
    stack = [(L, False)]
    order = []
    while stack:
        x, done = stack.pop()
        if x is None: continue
        if not done:
            stack += [(x, True), (x.right, False), (x.left, False)]
        else:
            cnt[x] = cnt[x.left] + cnt[x.right] + (x.val in targets); order.append(x)
    members = [x for x in order if cnt[x] > 0]          # gồm L
    return len(members) - 1

def steiner_brute(root, nodes):
    # hợp các đường đi từ LCA tới từng mục tiêu
    L = Solution().lowestCommonAncestor(root, nodes)
    parent = {L: None}; stack = [L]
    while stack:
        u = stack.pop()
        for c in (u.left, u.right):
            if c: parent[c] = u; stack.append(c)
    edges = set()
    for t in nodes:
        x = t
        while x is not L:
            edges.add((parent[x], x)); x = parent[x]
    return len(edges)

rnd = random.Random(5)
for _ in range(1500):
    rt, nodes_all = random_tree(rnd.randint(1, 20), rnd)
    ns = rnd.sample(nodes_all, rnd.randint(1, len(nodes_all)))
    assert lca_set_iter_sol(rt, ns) is lca_set_dfs(rt, ns)
    assert steiner_size_sol(rt, ns) == steiner_brute(rt, ns)
r, ix = build_tree(TREE)
print("Bài 4, Bài 5 OK | steiner({7, 6, 4}) =", steiner_size_sol(r, [ix[7], ix[6], ix[4]]), "cạnh")

Bài 4, Bài 5 OK | steiner({7, 6, 4}) = 4 cạnh


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "LCA của tập mục tiêu" (hậu thứ tự lặp, đếm):**

```python
targets = {u.val for u in nodes}; k = len(targets)
cnt = {None: 0}
stack = [(root, False)]
while stack:
    x, done = stack.pop()
    if x is None: continue
    if not done:
        stack += [(x, True), (x.right, False), (x.left, False)]
    else:
        c = cnt[x.left] + cnt[x.right] + (x.val in targets)
        if c == k: return x
        cnt[x] = c
```

**Template ngắn (mọi mục tiêu chắc chắn tồn tại, cây nông):**

```python
def dfs(x):
    if x is None or x.val in targets: return x
    l, r = dfs(x.left), dfs(x.right)
    return x if l and r else (l or r)
```

**Checklist:**
- [ ] Dùng `set` cho tập mục tiêu?
- [ ] Không dừng sớm khi dùng cách **đếm**?
- [ ] Chỉ nhận nút **đầu tiên** đạt `cnt = k`?
- [ ] `k = 1` trả về chính nút đó?
- [ ] Cây lệch `10⁴` nút: bản lặp?
- [ ] Đã đối chiếu với vét cạn trên cây ngẫu nhiên?
- [ ] Chữ ký đúng: `def lowestCommonAncestor(self, root: 'TreeNode', nodes: 'List[TreeNode]') -> 'TreeNode'`?